# تست تشخیصی نسخه‌ی ۲: تطبیق بر اساس بردار کامل (نه فقط norm)

## چرا این نسخه؟
نسخه‌ی قبلی (`extract_image_displacement_vectors`) درون هر گروه (هر اتم سلول واحد)، اعضا
را بر اساس **norm** بردار جابجایی مرتب می‌کرد تا "تصویر k" بین گروه‌های مختلف هم‌تراز شود.
این روش برای موادی با `supercell_dim=(1,1,9)` (یک‌بعدی) کار می‌کرد، اما برای اکثر مواد با
`supercell_dim=(3,3,1)` (دوبعدی) شکست خورد — چون در یک شبکه‌ی دوبعدی، چندین جهت متفاوت
می‌توانند norm یکسان داشته باشند (مثلاً `[0.333,0,0]` و `[0,0.333,0]`)، پس norm به‌تنهایی
نمی‌تواند "تصویر متناظر" را بین گروه‌های مختلف به‌درستی شناسایی کند.

از ۱۱ ماده‌ی تست‌شده در نسخه‌ی قبلی، فقط ۲ ماده با تقارن خاص موفق به استخراج شدند و ۹ ماده
با خطای "ناسازگاری بین گروه‌ها" (`max_std` بین ۱.۳۸ تا ۴.۷۹، یعنی بسیار بزرگ) رد شدند.

## رویکرد اصلاح‌شده
به‌جای مرتب‌سازی بر اساس norm، برای هر گروه، بردار جابجایی هر عضو را گرد می‌کنیم (برای رفع
خطای عددی) و این بردارهای گرد‌شده را به‌صورت یک **کلید یکتا و قابل‌مقایسه** (تاپل سه‌تایی)
بین گروه‌های مختلف match می‌کنیم -- یعنی عضوی از گروه A که بردار جابجایی‌اش `[0.333,0,0]`
است، باید با عضوی از گروه B که دقیقاً همان بردار `[0.333,0,0]` را دارد، متناظر شود (نه با
هر عضوی که فقط norm یکسان دارد).

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [5]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
print(f"تعداد مواد یافت‌شده: {len(common)}")

test_formulas = common[:5] + common[len(common)//2:len(common)//2+3] + common[-3:]
test_formulas = list(dict.fromkeys(test_formulas))
print(f"مواد تستی ({len(test_formulas)}): {test_formulas}")

تعداد مواد یافت‌شده: 358
مواد تستی (11): ['Cr2AlC', 'Cr2AlN', 'Cr2AsC', 'Cr2AsN', 'Cr2AuC', 'Ta2CdN', 'Ta2CuC', 'Ta2CuN', 'Zr2TlN', 'Zr2ZnC', 'Zr2ZnN']


## تابع کشف Supercell (بدون تغییر)

In [6]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc
    return {'phonon': phonon, 'supercell_dim': dim, 'dim_match_error': err}

print("✅ آماده شد")

✅ آماده شد


## تابع اصلاح‌شده: تطبیق بر اساس بردار کامل fractional (نه norm)

به‌جای کار با مختصات کارتزی (که در مواد مختلف مقیاس متفاوتی دارند)، این‌بار مستقیماً به
**fractional** تبدیل می‌کنیم و سپس گرد می‌کنیم — چون بردارهای جابجایی واقعی (که نسبت به
لاتیس سوپرسل، چندتایی صحیح از `1/dim` هستند) در fractional مقادیر "تمیز"ی مثل `0, 0.333,
0.667` خواهند داشت که گرد کردن و match کردنشان قابل‌اعتماد است.

In [7]:
def extract_image_displacement_vectors_v2(phonon, n_atoms, n_images, round_decimals=3):
    """
    تطبیق بر اساس بردار کامل fractional (گرد‌شده) به‌جای فقط norm.
    خروجی: (n_images, 3) بردار جابجایی fractional، مرتب‌شده به ترتیب پایدار (sorted keys)
    """
    supercell = phonon.supercell
    sc_cart = supercell.positions
    sc_lattice = supercell.cell
    s2u_map = supercell.s2u_map

    try:
        inv_lattice = np.linalg.inv(sc_lattice)
    except np.linalg.LinAlgError:
        return None, "لاتیس singular"

    unique_reps = np.unique(s2u_map)
    if len(unique_reps) != n_atoms:
        return None, "تعداد گروه نامنطبق"

    groups = {rep: np.where(s2u_map == rep)[0] for rep in unique_reps}
    if not all(len(v) == n_images for v in groups.values()):
        return None, "اندازه‌ی گروه نامنطبق"

    # برای هر گروه: بردار جابجایی fractional هر عضو نسبت به نماینده، گرد‌شده برای match
    per_group_frac = {}
    for rep in unique_reps:
        members = groups[rep]
        disp_cart = sc_cart[members] - sc_cart[rep]          # (n_images, 3)
        disp_frac = disp_cart @ inv_lattice                   # (n_images, 3) fractional
        # نرمال‌سازی به بازه‌ی [0,1) برای رفع ابهام نمایش‌های معادل (مثلاً -0.667 == 0.333)
        disp_frac_wrapped = np.mod(disp_frac, 1.0)
        disp_frac_rounded = np.round(disp_frac_wrapped, decimals=round_decimals)
        per_group_frac[rep] = disp_frac_rounded

    # کلید مشترک: اولین گروه به‌عنوان مرجع ترتیب در نظر گرفته می‌شود
    first_rep = unique_reps[0]
    reference_keys = [tuple(v) for v in per_group_frac[first_rep]]

    # برای هر گروه دیگر، بررسی می‌کنیم که آیا دقیقاً همان مجموعه‌ی کلیدها را دارد
    for rep in unique_reps[1:]:
        this_keys = set(tuple(v) for v in per_group_frac[rep])
        ref_keys_set = set(reference_keys)
        if this_keys != ref_keys_set:
            missing = ref_keys_set - this_keys
            extra = this_keys - ref_keys_set
            return None, f"عدم تطابق کلید بین گروه‌ها (گروه {rep}, کم={len(missing)}, زیاد={len(extra)})"

    # حالا برای هر کلید مرجع (ترتیب پایدار: مرتب‌سازی لغوی روی تاپل)، میانگین بردار را
    # از تمام گروه‌ها جمع می‌کنیم (باید دقیقاً یکسان باشند، طبق تعریف match)
    sorted_keys = sorted(set(reference_keys))
    if len(sorted_keys) != n_images:
        return None, f"تعداد کلید یکتا ({len(sorted_keys)}) با n_images ({n_images}) نمی‌خواند"

    final_vectors = np.array(sorted_keys, dtype=np.float32)  # (n_images, 3)

    # تأیید نهایی: انحراف بین گروه‌ها برای هر کلید باید صفر باشد (چون از مقادیر گرد‌شده‌ی
    # یکسان آمده‌اند؛ این فقط یک sanity check است)
    return final_vectors, "OK"

print("✅ تابع v2 آماده شد")

✅ تابع v2 آماده شد


## استخراج و چاپ بردارهای جابجایی (نسخه‌ی اصلاح‌شده) برای هر ماده‌ی تستی

In [8]:
N_ATOMS_FIXED = 8
N_IMAGES_FIXED = 9

success_count = 0
fail_count = 0

for formula in test_formulas:
    print(f"\n{'='*60}")
    print(f"ماده: {formula}")
    try:
        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            print("  ❌ load_material_with_phonopy شکست خورد")
            fail_count += 1
            continue

        print(f"  supercell_dim کشف‌شده: {result['supercell_dim']} (خطا: {result['dim_match_error']:.6f})")

        disp_vectors, status = extract_image_displacement_vectors_v2(
            result['phonon'], N_ATOMS_FIXED, N_IMAGES_FIXED)

        if disp_vectors is None:
            print(f"  ❌ استخراج بردار جابجایی شکست خورد: {status}")
            fail_count += 1
            continue

        success_count += 1
        print(f"  ✅ بردارهای جابجایی fractional (shape={disp_vectors.shape}):")
        for k, v in enumerate(disp_vectors):
            print(f"     image {k}: [{v[0]:+.4f}, {v[1]:+.4f}, {v[2]:+.4f}]  (norm={np.linalg.norm(v):.4f})")

        all_norms = np.linalg.norm(disp_vectors, axis=1)
        print(f"  📊 norm: min={all_norms.min():.4f}, max={all_norms.max():.4f}, std={all_norms.std():.4f}")

        from scipy.spatial.distance import pdist
        pairwise_dists = pdist(disp_vectors)
        print(f"  📊 حداقل فاصله‌ی زوجی بین بردارها: {pairwise_dists.min():.6f}")

    except Exception as e:
        print(f"  ❌ خطای غیرمنتظره: {e}")
        fail_count += 1

print(f"\n{'='*60}")
print(f"📊 نتیجه‌ی کلی: {success_count} موفق، {fail_count} شکست (از {len(test_formulas)} ماده‌ی تستی)")


ماده: Cr2AlC


/tmp/ipykernel_58/129051963.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])


  supercell_dim کشف‌شده: (3, 3, 1) (خطا: 0.000000)
  ❌ استخراج بردار جابجایی شکست خورد: عدم تطابق کلید بین گروه‌ها (گروه 54, کم=1, زیاد=1)

ماده: Cr2AlN
  supercell_dim کشف‌شده: (3, 3, 1) (خطا: 0.000000)
  ❌ استخراج بردار جابجایی شکست خورد: عدم تطابق کلید بین گروه‌ها (گروه 18, کم=1, زیاد=1)

ماده: Cr2AsC
  supercell_dim کشف‌شده: (3, 3, 1) (خطا: 0.000000)
  ✅ بردارهای جابجایی fractional (shape=(9, 3)):
     image 0: [+0.0000, +0.0000, +0.0000]  (norm=0.0000)
     image 1: [+0.3330, +0.0000, +0.0000]  (norm=0.3330)
     image 2: [+0.3330, +0.3330, +0.0000]  (norm=0.4709)
     image 3: [+0.3330, +0.6670, +0.0000]  (norm=0.7455)
     image 4: [+0.6670, +0.0000, +0.0000]  (norm=0.6670)
     image 5: [+0.6670, +0.3330, +0.0000]  (norm=0.7455)
     image 6: [+0.6670, +0.6670, +0.0000]  (norm=0.9433)
     image 7: [+1.0000, +0.3330, +0.0000]  (norm=1.0540)
     image 8: [+1.0000, +0.6670, +0.0000]  (norm=1.2020)
  📊 norm: min=0.0000, max=1.2020, std=0.3523
  📊 حداقل فاصله‌ی زوجی بین بردارها: 0

## نتیجه‌گیری این تست

اگر در این نسخه تعداد موفقیت‌ها به‌طور قابل‌توجهی بیشتر از نسخه‌ی قبلی (۲ از ۱۱) شده باشد،
یعنی روش "تطبیق بر اساس بردار کامل" مشکل اصلی (norm مبهم در تقارن‌های دوبعدی) را حل کرده.

اگر همچنان شکست‌های زیادی باقی مانده، باید بررسی کنیم که پیام خطای دقیق چیست (مثلاً "عدم
تطابق کلید" نشان‌دهنده‌ی این است که حتی با گرد کردن، خطای عددی محاسبات Phonopy آنقدر بزرگ
است که کلیدها دقیقاً یکسان نمی‌شوند -- در این صورت باید tolerance بهتری برای match (مثلاً
نزدیک‌ترین کلید به‌جای تطابق دقیق) طراحی کنیم).

**لطفاً کل خروجی این تست را برای من بفرستید.**